In [1]:
import requests
from bs4 import BeautifulSoup

print("requests OK:", requests.__version__)
print("BeautifulSoup OK")

requests OK: 2.33.1
BeautifulSoup OK


¿Qué estamos haciendo acá?

requests.get(url) → le manda una petición HTTP GET al servidor
response.status_code → el servidor responde con un código. 200 = éxito, 404 = no encontrado, 500 = error del servidor
response.text → el HTML crudo que devolvió el servidor, como un string gigante

In [2]:
url = "http://books.toscrape.com/"

response = requests.get(url)

print("Status code:", response.status_code)
print("Tipo de respuesta:", type(response.text))
print("Primeros 200 caracteres del HTML:")
print(response.text[:200])

Status code: 200
Tipo de respuesta: <class 'str'>
Primeros 200 caracteres del HTML:
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if I


Paso 3 — Parsear el HTML con BeautifulSoup
Tenemos el HTML como string gigante, pero así no podemos buscar nada fácilmente. BeautifulSoup lo convierte en un árbol navegable.
En una celda nueva:
¿Qué es el árbol HTML?
El HTML tiene una estructura jerárquica:
html
 └── body
      └── div
           └── h1 → "Books to Scrape"
           └── ul
                └── li → "Mystery"
                └── li → "Travel"
BeautifulSoup nos deja navegar esa estructura y buscar elementos por su etiqueta, clase CSS, o atributos.

In [3]:
soup = BeautifulSoup(response.text, "html.parser")

print("Tipo:", type(soup))
print("Título de la página:", soup.title.text)

Tipo: <class 'bs4.BeautifulSoup'>
Título de la página: 
    All products | Books to Scrape - Sandbox



Paso 4 — Encontrar las categorías
Abrí el sitio en el navegador: http://books.toscrape.com/
Hacé clic derecho sobre cualquier categoría del sidebar izquierdo → Inspeccionar. Vas a ver algo así:
<ul class="nav nav-list">
    <li>
        <a href="catalogue/category/books_1/index.html">Books</a>
        <ul>
            <li><a href="catalogue/category/books/travel_2/index.html">Travel</a></li>
            <li><a href="catalogue/category/books/mystery_3/index.html">Mystery</a></li>
            ...
        </ul>
    </li>
</ul>
¿Qué hace soup.select()?
Usa selectores CSS igual que en el navegador. ul.nav.nav-list li a significa: buscá todos los <a> que estén dentro de un <li> que esté dentro de un <ul> con clases nav y nav-list.

In [6]:
# Buscamos todos los <a> dentro del nav de categorías
# [1:] descarta el primero que es "Books" (categoría general, no nos interesa)
categorias = soup.select("ul.nav.nav-list li a")[1:]

print(f"Total de categorías: {len(categorias)}")
print("\nPrimeras 5:")
for cat in categorias[:5]:
    print(" -", cat.text.strip(), "→", cat["href"])

Total de categorías: 50

Primeras 5:
 - Travel → catalogue/category/books/travel_2/index.html
 - Mystery → catalogue/category/books/mystery_3/index.html
 - Historical Fiction → catalogue/category/books/historical-fiction_4/index.html
 - Sequential Art → catalogue/category/books/sequential-art_5/index.html
 - Classics → catalogue/category/books/classics_6/index.html


Paso 5 — Construir las URLs completas
Fijate que el href que nos devuelve es relativo: catalogue/category/books/travel_2/index.html
Para poder visitarlo necesitamos la URL completa: http://books.toscrape.com/catalogue/category/books/travel_2/index.html
En una celda nueva:
¿Qué hace urljoin?
Combina una URL base con una relativa y resuelve la ruta correcta automáticamente.
Es más seguro que concatenar strings con + porque maneja correctamente las barras y rutas relativas como ../.

In [7]:
from urllib.parse import urljoin

BASE_URL = "http://books.toscrape.com/"

# Probamos con la primera categoría
primera = categorias[0]

nombre = primera.text.strip()
url_completa = urljoin(BASE_URL, primera["href"])

print("Nombre:", nombre)
print("URL completa:", url_completa)

Nombre: Travel
URL completa: http://books.toscrape.com/catalogue/category/books/travel_2/index.html


Paso 6 — Entrar a una categoría y ver los libros
Ahora vamos a entrar a esa URL y buscar los libros que tiene. En una celda nueva:
¿Por qué miramos el HTML del primer libro?
Antes de extraer datos necesitamos entender la estructura. El .prettify() nos muestra el HTML indentado para leerlo más fácil. Con eso vamos a saber exactamente dónde está el título, precio y rating.

¿Por qué miramos el HTML del primer libro?
Antes de extraer datos necesitamos entender la estructura. El .prettify() nos muestra el HTML indentado para leerlo más fácil. Con eso vamos a saber exactamente dónde está el título, precio y rating.

Perfecto. ✅ Vemos la estructura del libro claramente.
Del HTML podemos identificar exactamente dónde está cada dato:

Título → <a href="..." > dentro del <h3> — tiene el atributo title con el nombre completo
Rating → <p class="star-rating Two"> — la segunda clase es el rating en palabras
Precio → todavía no se ve, está más abajo en el HTML

In [8]:
# Entramos a la categoría Travel
respuesta_cat = requests.get(url_completa)
soup_cat = BeautifulSoup(respuesta_cat.text, "html.parser")

# Cada libro está dentro de un <article class="product_pod">
libros = soup_cat.select("article.product_pod")

print(f"Libros encontrados en Travel: {len(libros)}")
print("\nPrimer libro (HTML resumido):")
print(libros[0].prettify()[:500])

Libros encontrados en Travel: 11

Primer libro (HTML resumido):
<article class="product_pod">
 <div class="image_container">
  <a href="../../../its-only-the-himalayas_981/index.html">
   <img alt="It's Only the Himalayas" class="thumbnail" src="../../../../media/cache/27/a5/27a53d0bb95bdd88288eaf66c9230d7e.jpg"/>
  </a>
 </div>
 <p class="star-rating Two">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="../../../its-only-the-hima


Paso 7 — Extraer datos del primer libro
En una celda nueva:

In [9]:
# Tomamos el primer libro para practicar
libro = libros[0]

# ── Título ──────────────────────────────────────────
titulo = libro.select_one("h3 a")["title"]

# ── Rating ──────────────────────────────────────────
# <p class="star-rating Two"> → ["star-rating", "Two"] → tomamos índice 1
rating_texto = libro.select_one("p.star-rating")["class"][1]

# Diccionario para convertir texto a número
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
rating = RATING_MAP[rating_texto]

# ── Precio ──────────────────────────────────────────
precio_texto = libro.select_one("p.price_color").text
precio = float(precio_texto.replace("£", "").replace("Â", "").strip())

print("Título:", titulo)
print("Rating:", rating, "estrellas")
print("Precio: £", precio)

Título: It's Only the Himalayas
Rating: 2 estrellas
Precio: £ 45.17


Paso 8 — Extraer todos los libros de la categoría
Ahora que sabemos cómo extraer datos de un libro, lo aplicamos a todos con un loop.
En una celda nueva:
Fijate algo interesante — algunos títulos tienen caracteres raros como Noahâs. Eso es un problema de encoding del sitio que vamos a ignorar por ahora, no afecta el funcionamiento.

In [10]:
libros_travel = []

for libro in libros:
    titulo = libro.select_one("h3 a")["title"]
    rating_texto = libro.select_one("p.star-rating")["class"][1]
    rating = RATING_MAP[rating_texto]
    precio_texto = libro.select_one("p.price_color").text
    precio = float(precio_texto.replace("£", "").replace("Â", "").strip())
    
    libros_travel.append({
        "titulo": titulo,
        "rating": rating,
        "precio": precio
    })

print(f"Total libros extraídos: {len(libros_travel)}")
print("\nTodos los libros:")
for l in libros_travel:
    print(f"  [{l['rating']}★] £{l['precio']:.2f} — {l['titulo']}")

Total libros extraídos: 11

Todos los libros:
  [2★] £45.17 — It's Only the Himalayas
  [4★] £49.43 — Full Moon over Noahâs Ark: An Odyssey to Mount Ararat and Beyond
  [3★] £48.87 — See America: A Celebration of Our National Parks & Treasured Sites
  [2★] £36.94 — Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel
  [3★] £37.33 — Under the Tuscan Sun
  [2★] £44.34 — A Summer In Europe
  [1★] £30.54 — The Great Railway Bazaar
  [4★] £56.88 — A Year in Provence (Provence #1)
  [1★] £23.21 — The Road to Little Dribbling: Adventures of an American in Britain (Notes From a Small Island #2)
  [3★] £38.95 — Neither Here nor There: Travels in Europe
  [5★] £26.08 — 1,000 Places to See Before You Die


Paso 9 — El problema de la paginación
Travel tiene solo 11 libros y caben en una página. Pero otras categorías tienen más libros y se dividen en múltiples páginas.
Mirá la URL de la segunda página de cualquier categoría grande:
http://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
El sitio tiene un botón "next" cuando hay más páginas. En una celda nueva:

In [11]:
# ¿Tiene Travel botón next?
next_btn = soup_cat.select_one("li.next a")
print("Botón next en Travel:", next_btn)

# Probemos con una categoría más grande: Mystery
url_mystery = urljoin(BASE_URL, "catalogue/category/books/mystery_3/index.html")
soup_mystery = BeautifulSoup(requests.get(url_mystery).text, "html.parser")

next_btn_mystery = soup_mystery.select_one("li.next a")
print("Botón next en Mystery:", next_btn_mystery)

if next_btn_mystery:
    print("URL siguiente página:", urljoin(url_mystery, next_btn_mystery["href"]))

Botón next en Travel: None
Botón next en Mystery: <a href="page-2.html">next</a>
URL siguiente página: http://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html


Paso 10 — Función para scrapear una categoría completa con paginación
Ahora convertimos todo lo que aprendimos en una función reutilizable. En una celda nueva:

In [12]:
import time

def scrapear_categoria(url_categoria):
    """
    Extrae todos los libros de una categoría manejando la paginación.
    Sigue el botón 'next' hasta que no haya más páginas.
    """
    libros = []
    url_actual = url_categoria
    pagina = 1

    while url_actual:  # mientras haya página siguiente
        print(f"  Scrapeando página {pagina}...")
        
        respuesta = requests.get(url_actual)
        soup = BeautifulSoup(respuesta.text, "html.parser")
        
        # Extraer libros de esta página
        articulos = soup.select("article.product_pod")
        
        for libro in articulos:
            titulo = libro.select_one("h3 a")["title"]
            rating_texto = libro.select_one("p.star-rating")["class"][1]
            rating = RATING_MAP[rating_texto]
            precio_texto = libro.select_one("p.price_color").text
            precio = float(precio_texto.replace("£", "").replace("Â", "").strip())
            
            libros.append({
                "titulo": titulo,
                "rating": rating,
                "precio": precio
            })
        
        # ¿Hay página siguiente?
        next_btn = soup.select_one("li.next a")
        if next_btn:
            url_actual = urljoin(url_actual, next_btn["href"])
            pagina += 1
        else:
            url_actual = None  # no hay más páginas, salir del while
        
        time.sleep(0.5)  # pausa entre páginas, scraping ético
    
    return libros


# Probamos con Mystery
print("Scrapeando Mystery...")
libros_mystery = scrapear_categoria(url_mystery)
print(f"\nTotal libros en Mystery: {len(libros_mystery)}")
print("\nPrimeros 3:")
for l in libros_mystery[:3]:
    print(f"  [{l['rating']}★] £{l['precio']:.2f} — {l['titulo']}")

Scrapeando Mystery...
  Scrapeando página 1...
  Scrapeando página 2...

Total libros en Mystery: 32

Primeros 3:
  [4★] £47.82 — Sharp Objects
  [1★] £19.63 — In a Dark, Dark Wood
  [4★] £56.50 — The Past Never Ends


Paso 11 — Scrapear TODAS las categorías
Ahora combinamos todo: la lista de categorías del paso 5 y la función del paso 10.
En una celda nueva:

In [ ]:
todos_los_libros = []

print(f"Scrapeando {len(categorias)} categorías...\n")

for i, cat in enumerate(categorias):
    nombre_cat = cat.text.strip()
    url_cat = urljoin(BASE_URL, cat["href"])
    
    print(f"[{i+1}/{len(categorias)}] {nombre_cat}")
    
    libros_cat = scrapear_categoria(url_cat)
    
    # Agregamos el nombre de categoría a cada libro
    for libro in libros_cat:
        libro["categoria"] = nombre_cat
    
    todos_los_libros.extend(libros_cat)
    print(f"  → {len(libros_cat)} libros\n")

print(f"✅ Scraping completo")
print(f"   Total libros: {len(todos_los_libros)}")
print(f"   Total categorías: {len(categorias)}")

Momento de reflexión — ¿qué construiste hasta acá?
requests.get(url)           → petición HTTP al servidor
BeautifulSoup(html, parser) → parsear el HTML en árbol navegable
soup.select("selector css") → buscar elementos por clase/etiqueta
urljoin(base, relativa)     → construir URLs absolutas
while url_actual:           → paginación automática
time.sleep(0.5)             → scraping ético

Perfecto. Vamos a crear las tablas.
En una celda nueva:

In [14]:
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

# SQLite no activa las foreign keys por defecto, hay que habilitarlas
cursor.execute("PRAGMA foreign_keys = ON")

print("✅ Conexión establecida")

✅ Conexión establecida


Perfecto. Ahora creamos las tablas una por una para que entiendas cada decisión.
Primero categories — la creamos primero porque books depende de ella:
¿Qué significa cada parte?

CREATE TABLE IF NOT EXISTS → si la tabla ya existe no falla, simplemente la ignora
PRIMARY KEY AUTOINCREMENT → SQLite genera el ID automáticamente: 1, 2, 3...
NOT NULL → ese campo no puede quedar vacío
UNIQUE → no pueden existir dos categorías con el mismo nombre o slug
conn.commit() → confirma los cambios en el archivo. Sin esto los cambios se pierden

In [15]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        name       TEXT    NOT NULL UNIQUE,
        slug       TEXT    NOT NULL UNIQUE
    )
""")

conn.commit()
print("✅ Tabla categories creada")

✅ Tabla categories creada


Ahora authors — la creamos antes que books porque book_author depende de ambas:
¿Qué es nuevo acá?

Los campos sin NOT NULL → pueden ser NULL. Eso es intencional porque estos datos vienen de la API y puede que no los encontremos
DEFAULT 'pending' → si no especificamos un valor para api_status, SQLite pone 'pending' automáticamente
DEFAULT CURRENT_TIMESTAMP → guarda automáticamente la fecha y hora en que se insertó el registro

In [16]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS authors (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        name              TEXT    NOT NULL UNIQUE,
        birth_year        INTEGER,
        country           TEXT,
        external_api_id   TEXT,
        total_known_works INTEGER,
        api_source        TEXT,
        api_status        TEXT DEFAULT 'pending',
        created_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

conn.commit()
print("✅ Tabla authors creada")

✅ Tabla authors creada


Ahora books:
¿Qué es nuevo acá?

REAL → tipo de dato para números decimales (el precio tiene centavos)
CHECK (rating BETWEEN 1 AND 5) → validación a nivel de base de datos. Si intentás insertar un rating de 6, SQLite rechaza el insert con error
REFERENCES categories(id) → esta es la foreign key. Le dice a SQLite que category_id debe existir en la tabla categories. No podés insertar un libro con una categoría que no existe

In [17]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        title       TEXT    NOT NULL,
        price       REAL    NOT NULL,
        rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
        category_id INTEGER NOT NULL REFERENCES categories(id),
        url         TEXT,
        description TEXT
    )
""")

conn.commit()
print("✅ Tabla books creada")

✅ Tabla books creada


Ahora la última tabla, book_author:
¿Qué es nuevo acá?

PRIMARY KEY (book_id, author_id) → clave primaria compuesta. En vez de un solo campo como ID, la combinación de los dos campos es única. Esto evita que el mismo par libro-autor se inserte dos veces
ON DELETE CASCADE → si borrás un libro de la tabla books, todas sus filas en book_author se borran automáticamente. Evita registros huérfanos (filas que apuntan a un libro que ya no existe)

In [18]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS book_author (
        book_id   INTEGER NOT NULL REFERENCES books(id)   ON DELETE CASCADE,
        author_id INTEGER NOT NULL REFERENCES authors(id) ON DELETE CASCADE,
        PRIMARY KEY (book_id, author_id)
    )
""")

conn.commit()
print("✅ Tabla book_author creada")

✅ Tabla book_author creada


Perfecto. Las 4 tablas están creadas. Vamos a verificar que todo quedó bien:

In [19]:
# Verificar que las tablas existen
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tablas = cursor.fetchall()

print("Tablas en la base de datos:")
for tabla in tablas:
    print(f"  ✅ {tabla[0]}")

Tablas en la base de datos:
  ✅ categories
  ✅ sqlite_sequence
  ✅ authors
  ✅ books
  ✅ book_author


Siguiente paso: insertar las categorías en la DB
En una celda nueva:

In [20]:
def insertar_categoria(nombre, slug):
    """
    Inserta una categoría y retorna su ID.
    INSERT OR IGNORE: si ya existe no falla, simplemente la ignora.
    """
    cursor.execute(
        "INSERT OR IGNORE INTO categories (name, slug) VALUES (?, ?)",
        (nombre, slug)
    )
    conn.commit()
    
    # Recuperamos el ID (sea nuevo o ya existente)
    cursor.execute("SELECT id FROM categories WHERE slug = ?", (slug,))
    return cursor.fetchone()[0]

# Probamos con Travel
id_travel = insertar_categoria("Travel", "travel_2")
print(f"✅ Travel insertada con ID: {id_travel}")

✅ Travel insertada con ID: 1


Ahora insertamos todas las categorías que scrapeamos:

In [21]:
def extraer_slug(href):
    """
    Extrae el slug de la URL de la categoría.
    Ejemplo: "catalogue/category/books/mystery_3/index.html" → "mystery_3"
    """
    return href.split("/")[-2]

# Insertar todas las categorías
print("Insertando categorías...")

for cat in categorias:
    nombre = cat.text.strip()
    slug   = extraer_slug(cat["href"])
    id_cat = insertar_categoria(nombre, slug)
    print(f"  ✅ {nombre} → ID: {id_cat}")

Insertando categorías...
  ✅ Travel → ID: 1
  ✅ Mystery → ID: 3
  ✅ Historical Fiction → ID: 4
  ✅ Sequential Art → ID: 5
  ✅ Classics → ID: 6
  ✅ Philosophy → ID: 7
  ✅ Romance → ID: 8
  ✅ Womens Fiction → ID: 9
  ✅ Fiction → ID: 10
  ✅ Childrens → ID: 11
  ✅ Religion → ID: 12
  ✅ Nonfiction → ID: 13
  ✅ Music → ID: 14
  ✅ Default → ID: 15
  ✅ Science Fiction → ID: 16
  ✅ Sports and Games → ID: 17
  ✅ Add a comment → ID: 18
  ✅ Fantasy → ID: 19
  ✅ New Adult → ID: 20
  ✅ Young Adult → ID: 21
  ✅ Science → ID: 22
  ✅ Poetry → ID: 23
  ✅ Paranormal → ID: 24
  ✅ Art → ID: 25
  ✅ Psychology → ID: 26
  ✅ Autobiography → ID: 27
  ✅ Parenting → ID: 28
  ✅ Adult Fiction → ID: 29
  ✅ Humor → ID: 30
  ✅ Horror → ID: 31
  ✅ History → ID: 32
  ✅ Food and Drink → ID: 33
  ✅ Christian Fiction → ID: 34
  ✅ Business → ID: 35
  ✅ Biography → ID: 36
  ✅ Thriller → ID: 37
  ✅ Contemporary → ID: 38
  ✅ Spirituality → ID: 39
  ✅ Academic → ID: 40
  ✅ Self Help → ID: 41
  ✅ Historical → ID: 42
  ✅ Christia

Ahora insertamos los libros:

In [22]:
def insertar_libro(titulo, precio, rating, category_id, url=""):
    """
    Inserta un libro y retorna su ID.
    """
    cursor.execute("""
        INSERT INTO books (title, price, rating, category_id, url)
        VALUES (?, ?, ?, ?, ?)
    """, (titulo, precio, rating, category_id, url))
    conn.commit()
    return cursor.lastrowid

# Insertar todos los libros
print("Insertando libros...")

for libro in todos_los_libros:
    # Necesitamos el ID de la categoría del libro
    cursor.execute(
        "SELECT id FROM categories WHERE name = ?",
        (libro["categoria"],)
    )
    resultado = cursor.fetchone()
    
    if resultado:
        category_id = resultado[0]
        insertar_libro(
            libro["titulo"],
            libro["precio"],
            libro["rating"],
            category_id
        )

# Verificar
cursor.execute("SELECT COUNT(*) FROM books")
total = cursor.fetchone()[0]
print(f"✅ Libros insertados: {total}")

Insertando libros...
✅ Libros insertados: 1000


Vamos a verificar que todo quedó bien con una consulta rápida:

In [23]:
# Ver una muestra de libros con su categoría
cursor.execute("""
    SELECT b.title, b.price, b.rating, c.name
    FROM books b
    JOIN categories c ON b.category_id = c.id
    LIMIT 5
""")

resultados = cursor.fetchall()

print("Muestra de libros en la DB:")
print("-" * 65)
for row in resultados:
    print(f"  [{row[2]}★] £{row[1]:.2f} — {row[0][:40]} ({row[3]})")

Muestra de libros en la DB:
-----------------------------------------------------------------
  [2★] £45.17 — It's Only the Himalayas (Travel)
  [4★] £49.43 — Full Moon over Noahâs Ark: An Odyssey  (Travel)
  [3★] £48.87 — See America: A Celebration of Our Nation (Travel)
  [2★] £36.94 — Vagabonding: An Uncommon Guide to the Ar (Travel)
  [3★] £37.33 — Under the Tuscan Sun (Travel)


Siguiente etapa: la API de Open Library
Vamos a buscar el autor de cada libro por título. En una celda nueva primero probamos con un solo libro para entender la respuesta de la API:

El endpoint que iba a usar (/search.json por título) nos da el autor pero no nos da los datos del autor — año de nacimiento, país, cantidad de obras. Para eso necesitamos el endpoint específico de autores.
El flujo correcto es en dos pasos:
1. Buscar libro por título → nos devuelve el nombre del autor
2. Buscar autor por nombre → nos devuelve birth_year, country, works, etc.
Ambos endpoints de Open Library:
Paso 1: https://openlibrary.org/search.json?title=Sharp+Objects&limit=1
Paso 2: https://openlibrary.org/search/authors.json?q=Gillian+Flynn&limit=1

Antes de escribir código, ¿entendés por qué necesitamos dos llamadas y no una sola? Quiero asegurarme de que cuando te pregunten en la presentación puedas explicarlo.

In [24]:
 # PASO 1: buscar libro por título para obtener el nombre del autor
titulo_prueba = "Sharp Objects"

url_buscar_libro = "https://openlibrary.org/search.json"
params = {"title": titulo_prueba, "limit": 1}

respuesta = requests.get(url_buscar_libro, params=params, timeout=10)
data = respuesta.json()

print(f"Status: {respuesta.status_code}")
print(f"Resultados encontrados: {data['numFound']}")

if data['numFound'] > 0:
    doc = data['docs'][0]
    print(f"\nTítulo:  {doc.get('title')}")
    print(f"Autor:   {doc.get('author_name')}")
    print(f"Año pub: {doc.get('first_publish_year')}")

Status: 200
Resultados encontrados: 46

Título:  Sharp Objects
Autor:   ['Gillian Flynn']
Año pub: 2006


In [25]:
# PASO 2: buscar datos del autor por nombre
autor_nombre = data['docs'][0].get('author_name')[0]  # "Gillian Flynn"

url_buscar_autor = "https://openlibrary.org/search/authors.json"
params_autor = {"q": autor_nombre, "limit": 1}

respuesta_autor = requests.get(url_buscar_autor, params=params_autor, timeout=10)
data_autor = respuesta_autor.json()

print(f"Status: {respuesta_autor.status_code}")
print(f"Resultados: {data_autor['numFound']}")

if data_autor['numFound'] > 0:
    doc_autor = data_autor['docs'][0]
    print(f"\nNombre:      {doc_autor.get('name')}")
    print(f"ID externo:  {doc_autor.get('key')}")
    print(f"Nacimiento:  {doc_autor.get('birth_date')}")
    print(f"País:        {doc_autor.get('top_subjects')}")
    print(f"Obras:       {doc_autor.get('work_count')}")

Status: 200
Resultados: 5

Nombre:      Gillian Flynn
ID externo:  OL1433006A
Nacimiento:  1971-02-24
País:        ['Fiction', 'Fiction, suspense', 'Fiction, thrillers, suspense', 'New York Times bestseller', 'Missouri', 'Crimes against', 'Women journalists, fiction', 'Thrillers', 'Missouri, fiction', 'Large type books']
Obras:       41


In [26]:
# Buscar detalle del autor usando su ID
autor_id = doc_autor.get('key')  # "OL1433006A"

url_detalle = f"https://openlibrary.org/authors/{autor_id}.json"
respuesta_detalle = requests.get(url_detalle, timeout=10)
data_detalle = respuesta_detalle.json()

print(f"Status: {respuesta_detalle.status_code}")
print(f"\nNombre:       {data_detalle.get('name')}")
print(f"Nacimiento:   {data_detalle.get('birth_date')}")
print(f"Birth place:  {data_detalle.get('birth_place')}")
print(f"Bio:          {str(data_detalle.get('bio', ''))[:100]}")

Status: 200

Nombre:       Gillian Flynn
Nacimiento:   1971-02-24
Birth place:  None
Bio:          {'type': '/type/text', 'value': "Flynn, who lives in Chicago, grew up in Kansas City, Missouri. She 


In [27]:
# Probamos con un autor más clásico
url_buscar_autor = "https://openlibrary.org/search/authors.json"
params_autor = {"q": "George Orwell", "limit": 1}

respuesta = requests.get(url_buscar_autor, params=params_autor, timeout=10)
doc = respuesta.json()['docs'][0]

autor_id = doc.get('key')
url_detalle = f"https://openlibrary.org/authors/{autor_id}.json"
detalle = requests.get(url_detalle, timeout=10).json()

print(f"Nombre:      {detalle.get('name')}")
print(f"Nacimiento:  {detalle.get('birth_date')}")
print(f"Birth place: {detalle.get('birth_place')}")
print(f"Obras:       {doc.get('work_count')}")

Nombre:      George Orwell
Nacimiento:  25 June 1903
Birth place: None
Obras:       683


In [32]:
url_buscar_autor = "https://openlibrary.org/search/authors.json"
params = {"q": "Jane Austen", "limit": 1}

respuesta = requests.get(url_buscar_autor, params=params, timeout=10)
doc = respuesta.json()['docs'][0]

autor_id = doc.get('key')
url_detalle = f"https://openlibrary.org/authors/{autor_id}.json"
detalle = requests.get(url_detalle, timeout=10).json()

print(f"Nombre:      {detalle.get('name')}")
print(f"Nacimiento:  {detalle.get('birth_date')}")
print(f"Birth place: {detalle.get('birth_place')}")
print(f"Obras:       {doc.get('work_count')}")

Nombre:      Jane Austen
Nacimiento:  December 16, 1775
Birth place: None
Obras:       2209
